# ABC2Vec — Notebook 10: Folk Tune Evolution Visualization

Uses BFDB (Broadside Folk Database) dated tunes to visualize the evolution/genealogy
of folk melodies over time.

**Key idea:** Embed dated tunes, project to 2D, and visualize how the folk tune
landscape shifts over decades/centuries.

**Analyses:**
1. UMAP colored by decade
2. Embedding drift over time
3. Tune type distribution changes over time
4. Interactive timeline visualization
5. Genealogy: trace melodic families through time

In [ ]:
import os, sys, json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns

try:
    import umap
except ImportError:
    !pip install umap-learn
    import umap

try:
    import plotly.express as px
    import plotly.graph_objects as go
    HAS_PLOTLY = True
except ImportError:
    !pip install plotly
    import plotly.express as px
    import plotly.graph_objects as go
    HAS_PLOTLY = True

PROJECT_DIR = Path('/Volumes/LLModels/ABC2VEC')
PROCESSED_DIR = PROJECT_DIR / 'data' / 'processed'
RESULTS_DIR = PROJECT_DIR / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_DIR))

device = torch.device('cuda' if torch.cuda.is_available() else 
                       'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Device: {device}")

## 10.1 Load BFDB Dated Tunes

In [ ]:
# Load BFDB data with year metadata
bfdb_path = PROCESSED_DIR / 'bfdb.json'
if bfdb_path.exists():
    with open(bfdb_path) as f:
        bfdb_data = json.load(f)
    bfdb_df = pd.DataFrame(bfdb_data)
    print(f"BFDB: {len(bfdb_df)} tunes")
    print(f"Columns: {list(bfdb_df.columns)}")
else:
    raise FileNotFoundError("Run notebook 01 first to download BFDB.")

# Filter to tunes with year metadata
if 'year' in bfdb_df.columns:
    dated_df = bfdb_df[bfdb_df['year'].notna()].copy()
    dated_df['year'] = dated_df['year'].astype(int)
    # Filter reasonable range
    dated_df = dated_df[(dated_df['year'] >= 1600) & (dated_df['year'] <= 2020)]
    dated_df['decade'] = (dated_df['year'] // 10) * 10
    print(f"\nDated tunes: {len(dated_df)}")
    print(f"Year range: {dated_df['year'].min()} – {dated_df['year'].max()}")
    print(f"\nDecade distribution:")
    print(dated_df['decade'].value_counts().sort_index().to_string())
else:
    print("No 'year' column — attempting to extract from metadata")
    # Try to extract year from other fields
    dated_df = bfdb_df.copy()
    # Create synthetic decades for visualization
    dated_df['decade'] = 0
    print(f"Using all {len(dated_df)} BFDB tunes (no date ordering)")

## 10.2 Encode Dated Tunes

In [ ]:
from abc2vec.model import ABC2VecModel, ABC2VecConfig
from abc2vec.tokenizer import ABCVocabulary, BarPatchifier

vocab = ABCVocabulary()
vocab_path = PROJECT_DIR / 'data' / 'processed' / 'vocabulary.json'
if vocab_path.exists():
    vocab.load(vocab_path)

patchifier = BarPatchifier(vocab, max_bar_length=64, max_bars=64)

config = ABC2VecConfig(
    vocab_size=len(vocab),
    max_bar_len=64, max_bars=64,
    d_model=256, n_heads=8, n_layers=6,
    d_ff=1024, d_embed=128, dropout=0.1,
)
model = ABC2VecModel(config).to(device)

ckpt_path = PROJECT_DIR / 'checkpoints' / 'best_model.pt'
if ckpt_path.exists():
    checkpoint = torch.load(ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

@torch.no_grad()
def encode_tunes(abc_texts, model, patchifier, device, batch_size=64):
    model.eval()
    all_emb = []
    for i in tqdm(range(0, len(abc_texts), batch_size), desc="Encoding"):
        batch = abc_texts[i:i+batch_size]
        bar_ids_list, bar_mask_list = [], []
        for text in batch:
            bars = patchifier.patchify(text)
            ids, mask = patchifier.encode(bars)
            bar_ids_list.append(ids)
            bar_mask_list.append(mask)
        bar_ids = torch.stack(bar_ids_list).to(device)
        bar_mask = torch.stack(bar_mask_list).to(device)
        emb = model.get_embedding(bar_ids, bar_mask)
        all_emb.append(emb.cpu().numpy())
    return np.concatenate(all_emb, axis=0)

print("Encoding dated tunes...")
dated_emb = encode_tunes(dated_df['abc_body'].tolist(), model, patchifier, device)
print(f"Embeddings: {dated_emb.shape}")

## 10.3 UMAP Colored by Decade

In [ ]:
print("Computing UMAP...")
reducer = umap.UMAP(n_components=2, n_neighbors=30, min_dist=0.3,
                     metric='cosine', random_state=42)
umap_2d = reducer.fit_transform(dated_emb)

dated_df_plot = dated_df.copy()
dated_df_plot['umap_x'] = umap_2d[:, 0]
dated_df_plot['umap_y'] = umap_2d[:, 1]

# Static plot colored by year
fig, ax = plt.subplots(figsize=(14, 10))
sc = ax.scatter(umap_2d[:, 0], umap_2d[:, 1],
                c=dated_df['year'].values, cmap='viridis',
                s=10, alpha=0.6)
plt.colorbar(sc, label='Year', ax=ax)
ax.set_title('Folk Tune Evolution — UMAP Colored by Year', fontsize=14)
ax.set_xticks([])
ax.set_yticks([])

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'evolution_umap_year.png', dpi=150, bbox_inches='tight')
plt.show()

## 10.4 Evolution Timeline: Centroid Drift

In [ ]:
# Track how the centroid of tune embeddings moves through time
decades = sorted(dated_df['decade'].unique())
# Filter decades with enough tunes
decade_counts = dated_df['decade'].value_counts()
valid_decades = [d for d in decades if decade_counts.get(d, 0) >= 5]

centroids_umap = []
centroids_emb = []
centroid_decades = []
centroid_sizes = []

for decade in valid_decades:
    mask = dated_df['decade'].values == decade
    n = mask.sum()
    
    centroid_emb = dated_emb[mask].mean(axis=0)
    centroid_umap = umap_2d[mask].mean(axis=0)
    
    centroids_emb.append(centroid_emb)
    centroids_umap.append(centroid_umap)
    centroid_decades.append(decade)
    centroid_sizes.append(n)

centroids_umap = np.array(centroids_umap)
centroids_emb = np.array(centroids_emb)

# Plot centroid trajectory on UMAP
fig, ax = plt.subplots(figsize=(14, 10))

# Background: all points lightly
ax.scatter(umap_2d[:, 0], umap_2d[:, 1], c='lightgray', s=5, alpha=0.3)

# Centroid trajectory
colors = cm.plasma(np.linspace(0, 1, len(centroid_decades)))
for i in range(len(centroids_umap)):
    ax.scatter(centroids_umap[i, 0], centroids_umap[i, 1],
               c=[colors[i]], s=centroid_sizes[i] * 3 + 50,
               edgecolors='black', linewidth=0.5, zorder=5)
    ax.annotate(f"{centroid_decades[i]}s", 
                (centroids_umap[i, 0], centroids_umap[i, 1]),
                fontsize=7, ha='center', va='bottom',
                xytext=(0, 5), textcoords='offset points')

# Connect centroids with arrows
for i in range(len(centroids_umap) - 1):
    ax.annotate('', xy=centroids_umap[i+1], xytext=centroids_umap[i],
                arrowprops=dict(arrowstyle='->', color='red', lw=1.5))

ax.set_title('Folk Tune Centroid Drift Over Decades', fontsize=14)
ax.set_xticks([])
ax.set_yticks([])

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'evolution_centroid_drift.png', dpi=150, bbox_inches='tight')
plt.show()

## 10.5 Embedding Distance Over Time

In [ ]:
# Measure how embedding centroids evolve: cosine distance between consecutive decades
from sklearn.metrics.pairwise import cosine_similarity

# Pairwise cosine similarity between decade centroids
centroid_sim = cosine_similarity(centroids_emb)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Heatmap of decade-decade similarity
sns.heatmap(centroid_sim, xticklabels=[f"{d}s" for d in centroid_decades],
            yticklabels=[f"{d}s" for d in centroid_decades],
            cmap='RdYlBu_r', vmin=0, vmax=1, ax=axes[0])
axes[0].set_title('Decade Centroid Cosine Similarity')

# Consecutive decade drift
consec_dist = []
for i in range(len(centroids_emb) - 1):
    cos_sim = cosine_similarity(
        centroids_emb[i:i+1], centroids_emb[i+1:i+2]
    )[0, 0]
    consec_dist.append(1 - cos_sim)  # Distance = 1 - similarity

axes[1].plot([f"{centroid_decades[i]}→{centroid_decades[i+1]}" 
              for i in range(len(consec_dist))],
             consec_dist, 'o-', color='steelblue', linewidth=2)
axes[1].set_xlabel('Decade Transition')
axes[1].set_ylabel('Cosine Distance')
axes[1].set_title('Consecutive Decade Centroid Drift')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'evolution_decade_drift.png', dpi=150, bbox_inches='tight')
plt.show()

## 10.6 Intra-Decade Diversity

In [ ]:
# How diverse are tunes within each decade?
# Measure as average pairwise cosine distance within each decade

decade_diversity = []
for decade in valid_decades:
    mask = dated_df['decade'].values == decade
    emb_subset = dated_emb[mask]
    if len(emb_subset) < 2:
        continue
    
    # Sample if too large
    if len(emb_subset) > 200:
        idx = np.random.choice(len(emb_subset), 200, replace=False)
        emb_subset = emb_subset[idx]
    
    sim = cosine_similarity(emb_subset)
    triu = np.triu_indices(len(emb_subset), k=1)
    mean_sim = sim[triu].mean()
    std_sim = sim[triu].std()
    
    decade_diversity.append({
        'decade': decade,
        'n_tunes': mask.sum(),
        'mean_similarity': mean_sim,
        'std_similarity': std_sim,
        'diversity': 1 - mean_sim,  # Higher = more diverse
    })

diversity_df = pd.DataFrame(decade_diversity)

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(diversity_df['decade'].astype(str), diversity_df['diversity'],
       color='steelblue', edgecolor='white')
ax.set_xlabel('Decade')
ax.set_ylabel('Intra-Decade Diversity (1 - mean cosine sim)')
ax.set_title('Melodic Diversity Within Each Decade')
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'evolution_diversity.png', dpi=150, bbox_inches='tight')
plt.show()

print(diversity_df.to_string(index=False))

## 10.7 Interactive Plotly Timeline

In [ ]:
if HAS_PLOTLY:
    # Interactive scatter: hover to see tune name, year, type
    plot_df = dated_df_plot.copy()
    plot_df['title_str'] = plot_df.get('title', plot_df.get('tune_name', 'Unknown'))
    plot_df['type_str'] = plot_df.get('tune_type', 'unknown')
    
    fig = px.scatter(
        plot_df, x='umap_x', y='umap_y',
        color='year',
        hover_data=['title_str', 'year', 'type_str'],
        title='Folk Tune Evolution — Interactive UMAP',
        color_continuous_scale='Viridis',
        opacity=0.6,
    )
    fig.update_traces(marker=dict(size=4))
    fig.update_layout(
        xaxis=dict(showticklabels=False, title=''),
        yaxis=dict(showticklabels=False, title=''),
        width=1000, height=800,
    )
    
    fig.write_html(RESULTS_DIR / 'evolution_interactive.html')
    fig.show()
    print(f"Interactive plot saved to {RESULTS_DIR / 'evolution_interactive.html'}")
else:
    print("Plotly not available — skipping interactive plot.")

## 10.8 Tune Type Distribution Over Time

In [ ]:
if 'tune_type' in dated_df.columns:
    # How tune types change in popularity over time
    type_decade = dated_df.groupby(['decade', 'tune_type']).size().unstack(fill_value=0)
    # Normalize per decade
    type_decade_pct = type_decade.div(type_decade.sum(axis=1), axis=0)
    
    # Keep only types with > 1% overall
    major_types = type_decade.sum().nlargest(6).index.tolist()
    type_decade_pct = type_decade_pct[major_types]
    
    fig, ax = plt.subplots(figsize=(14, 6))
    type_decade_pct.plot(kind='area', stacked=True, ax=ax, alpha=0.8)
    ax.set_xlabel('Decade')
    ax.set_ylabel('Proportion')
    ax.set_title('Tune Type Distribution Over Time')
    ax.legend(loc='upper left', bbox_to_anchor=(1, 1))
    
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'evolution_type_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("No tune_type column available for temporal analysis.")